# Notebook 01: Foundation & AWS Setup

## Learning Objectives
- Set up AWS environment for AgentCore
- Understand AgentCore architecture
- Validate development environment
- Configure basic authentication

## Prerequisites
- AWS account with appropriate permissions
- Deno 2.9+ (see `.deno-version`), with the Deno Jupyter kernel installed
- API keys for external services

**⚠️ IMPORTANT: Run `./setup.sh` from project root before starting!**

This notebook runs TypeScript on the Deno kernel. In VS Code, pick the **Deno** kernel in the top right.


## Step 1: Environment Setup

In [ ]:
// Verify environment setup
const setupDone = await Deno.stat("../../deno.json").then(() => true).catch(() => false);
if (!setupDone) {
  console.log("❌ Please run ./setup.sh from project root first!");
} else {
  console.log(`✅ Working directory: ${Deno.cwd()}`);
  console.log("✅ Environment setup verified");
}

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");
// Remove any existing credential env vars to force profile usage
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

console.log("✅ AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import { defaultProvider } from "@aws-sdk/credential-provider-node";
import { loadEnv } from "../shared/notebook.ts";

// Load environment variables
await loadEnv();

// Verify AWS credentials
const credentials = await defaultProvider()();
console.log(`AWS Access Key: ${credentials.accessKeyId.slice(0, 8)}...`);
console.log(`AWS Region: ${Deno.env.get("AWS_REGION") ?? "us-west-2"}`);

## Step 2: API Keys Validation

In [ ]:
import { maskKey } from "../shared/notebook.ts";

// Validate API keys
const apiKeys = {
  AVIATIONSTACK_API_KEY: Deno.env.get("AVIATIONSTACK_API_KEY"),
  OPENWEATHERMAP_API_KEY: Deno.env.get("OPENWEATHERMAP_API_KEY"),
  EXCHANGERATE_API_KEY: Deno.env.get("EXCHANGERATE_API_KEY"),
};

console.log("🔑 API Key Status:");
for (const [keyName, keyValue] of Object.entries(apiKeys)) {
  console.log(`${keyValue ? "✅" : "❌"} ${keyName}: ${maskKey(keyValue)}`);
}

const missingKeys = Object.entries(apiKeys).filter(([, v]) => !v).map(([k]) => k);
if (missingKeys.length > 0) {
  console.log(`\n⚠️ Missing API keys: ${missingKeys.join(", ")}`);
  console.log("Please configure these before proceeding.");
} else {
  console.log("\n✅ All API keys configured!");
}

## Step 3: Environment Validation

In [ ]:
import { BedrockClient, ListFoundationModelsCommand } from "@aws-sdk/client-bedrock";
import { GetCallerIdentityCommand, STSClient } from "@aws-sdk/client-sts";
import { sh } from "../shared/notebook.ts";

// Comprehensive environment check
async function checkEnvironment(): Promise<[string, string][]> {
  const checks: [string, string][] = [];

  // Deno version (the project pins it in .deno-version)
  const [major, minor] = Deno.version.deno.split(".").map(Number);
  const denoOk = major > 2 || (major === 2 && minor >= 9);
  checks.push([denoOk ? "✅" : "❌", `Deno ${Deno.version.deno}${denoOk ? "" : " (need 2.9+)"}`]);

  // Deno Jupyter kernel
  const kernel = await sh("deno", ["jupyter"], { quiet: true });
  const kernelInstalled = `${kernel.stdout}${kernel.stderr}`.includes("already installed");
  checks.push([
    kernelInstalled ? "✅" : "❌",
    kernelInstalled ? "Deno Jupyter kernel installed" : "Deno Jupyter kernel missing (run ./setup.sh)",
  ]);

  // AWS credentials
  try {
    const identity = await new STSClient({}).send(new GetCallerIdentityCommand({}));
    checks.push(["✅", `AWS Account: ${identity.Account}`]);
  } catch (error) {
    checks.push(["❌", `AWS credentials: ${error}`]);
  }

  // Bedrock access
  try {
    await new BedrockClient({ region: "us-west-2" }).send(new ListFoundationModelsCommand({}));
    checks.push(["✅", "Bedrock access"]);
  } catch (error) {
    checks.push(["❌", `Bedrock access: ${error}`]);
  }

  return checks;
}

console.log("🔍 Environment Validation:");
const validationResults = await checkEnvironment();
for (const [status, message] of validationResults) {
  console.log(`${status} ${message}`);
}

// Summary
const passed = validationResults.filter(([status]) => status === "✅").length;
const total = validationResults.length;
console.log(`\n📊 Validation Summary: ${passed}/${total} checks passed`);

console.log(
  passed === total
    ? "🎉 Environment ready! Proceed to Notebook 02."
    : "⚠️ Please fix the issues above before continuing.",
);

## Next Steps

✅ **Completed in this notebook:**
- AWS environment setup and validation
- AgentCore introduction and concepts
- API keys configuration check
- Development environment verification

➡️ **Next: Notebook 02 - Runtime Setup**
- Configure AgentCore Runtime
- Create basic conversational agent
- Implement multi-turn dialogue
- Test conversation flow
